# 12 — Qwen3.5-2B on a free Colab GPU

**In one sentence:** this notebook asks a small 2026 model to solve code problems, twice —
once with its *thinking* switched ON and once with it OFF — and counts how many tokens that costs.

**Why this notebook exists.** Everything used to run on a Mac. One medium problem took
**250 seconds**. A free Colab T4 answers 16 problems at the same time, so it is much faster.
Qwen3.5-2B needs about 4.5 GB and the T4 has about 15 GB, so the whole model fits on the GPU.
(Our old model did not fit, and that is why it crawled at 4.4 tokens/s — `DECISIONS #43, #46`.)

**Where we are:** `PROBLEM ✅ → GAP ✅ → QUESTION ✅ → HYPOTHESIS ✅ → EXPERIMENT ⬅ HERE`

**Before you run anything:** menu **Runtime → Change runtime type → T4 GPU**.

Run the cells in order. Cells 3 and 6 are **safety nets** — if they fail, stop. Do not skip them.

## 1. Install what we need
**Problem:** Colab does not have our libraries. **Why:** without them nothing runs.
**In:** nothing. **Out:** installed packages. **Why this way:** plain `transformers` for asking
questions (we only add Unsloth later, for training), plus `evalplus` which holds the HumanEval
problems *and their tests*.

In [ ]:
%%capture
!pip install -q -U "transformers>=5.5.0" accelerate
!pip install -q evalplus

In [ ]:
# Check it worked, and that we really have a GPU. If this says "No GPU", fix the runtime type.
import torch, transformers
print("transformers", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU  <-- STOP")
print("bfloat16 supported:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else "-")
print("GPU memory:", f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB"
      if torch.cuda.is_available() else "-")

## 2. Get the code, and a safe place to save answers
**Problem:** a free session can die at any second, and then a finished answer is lost.
**Why:** answers cost GPU time; losing one means paying for it twice.
**In:** our public GitHub repo. **Out:** the code, and a results folder on Google Drive.
**Why this way:** Drive outlives the session. The results file is **append-only** and we
`fsync` after every batch, so a crash can never cost more than the batch being written.

In [ ]:
import os, sys

REPO = "https://github.com/mahmudulhaquequdrati/stop-overthinking-thesis.git"
if not os.path.isdir("/content/thesis"):
    !git clone -q {REPO} /content/thesis
else:
    !cd /content/thesis && git pull -q
os.chdir("/content/thesis")
sys.path.insert(0, "/content/thesis/scripts")

try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = "/content/drive/MyDrive/stop-overthinking/results"
except Exception as e:
    print("No Google Drive (fine when testing locally):", e)
    OUT_DIR = "results"
os.makedirs(OUT_DIR, exist_ok=True)
print("code in :", os.getcwd())
print("saving to:", OUT_DIR)

## 3. ⚠️ SAFETY NET — is the thinking counter correct?
**Problem:** Qwen and our old model mark their thinking differently.

```
OLD (Gemma)  prompt ends: ...            output: <|channel>thought ... <channel|> ANSWER
                                                 ^^^^^^ the START marker is in the OUTPUT

NEW (Qwen)   prompt ends: ... <think>    output: thinking ... </think> ANSWER
                              ^^^^^^^ the START marker is in the PROMPT, not the output
```

Code that looks for the start marker finds **nothing** with Qwen. It would report **0 thinking
tokens on every answer** — and nothing would crash. Thinking tokens are the number this whole
thesis is about, so that mistake would quietly ruin everything.

**In:** nothing (only the tokenizer, a few MB — no GPU needed). **Out:** PASS or FAIL.
**Why this way:** it tests the **real** chat template downloaded from Hugging Face, not a copy
we typed by hand. **If anything says FAIL, stop here.**

In [ ]:
!python scripts/test_prompts.py

## 4. Load the model
**Problem:** we need the model on the GPU. **Why:** everything else waits on this.
**In:** `unsloth/Qwen3.5-2B` (4.58 GB). **Out:** a model and a tokenizer.
**Why this way:** 16-bit, straight onto the GPU, **no CPU offloading** — offloading is exactly
what made our old model 12× slower. A T4 has no `bfloat16`, so the code falls back to `float16`.

👉 **Check the printed memory: it should be about 5 GB.** If it is near 14 GB, something loaded
in the wrong precision — stop and ask.

In [ ]:
import models, prompts
from gen_colab import load_model

profile = models.get("qwen35_2b")
model, tok = load_model(profile)

# Show that the switch really changes the prompt.
on, off = prompts.check_prompt_has_switch(tok, profile)
print("\nthinking ON  prompt ends:", repr(on[-40:]))
print("thinking OFF prompt ends:", repr(off[-40:]))

## 5. Get the problems
**Problem:** every run used to download the benchmarks again. **Why:** slow, and it needs
internet mid-run. **In:** HumanEval (easy) + LiveCodeBench (easy and medium).
**Out:** one local file `data/problems.json` with the questions *and* their tests.
**Why this way:** written once, only read afterwards, so a run can never change the problems
under our feet.

In [ ]:
!python scripts/build_problem_set.py

## 6. ⚠️ SAFETY NET — a 2-problem smoke test
**Problem:** a 2-hour run that was wrong from the first second is the most expensive mistake
we can make. **Why:** we check the two things that silently break, on 2 problems, in ~1 minute.
**In:** 2 problems. **Out:** two token counts.

👉 **What you must see:**
- `thinking_on` → thinking tokens in the **hundreds**. If it prints **0**, the trap from cell 3
  is still there. **Stop.**
- `thinking_off` → thinking tokens **exactly 0** on every row. If not, the switch is being ignored.

In [ ]:
SMOKE = f"{OUT_DIR}/smoke-test.jsonl"
!rm -f "{SMOKE}"
!python scripts/gen_colab.py --policy thinking_on  --n 2 --batch 2 --max-tokens 1024 --out "{SMOKE}"
!python scripts/gen_colab.py --policy thinking_off --n 2 --batch 2 --max-tokens 1024 --out "{SMOKE}"

In [ ]:
import json
print(f"{'policy':<14}{'thinking':>10}{'total':>8}   first 60 characters of the answer")
for line in open(SMOKE):
    r = json.loads(line)
    print(f"{r['policy']:<14}{r['thinking_tokens']:>10}{r['total_new_tokens']:>8}   "
          f"{r['answer_text'][:60].strip()!r}")

rows = [json.loads(l) for l in open(SMOKE)]
on  = [r for r in rows if r["policy"] == "thinking_on"]
off = [r for r in rows if r["policy"] == "thinking_off"]
ok = all(r["thinking_tokens"] > 0 for r in on) and all(r["thinking_tokens"] == 0 for r in off)
print("\n" + ("SMOKE TEST PASSED - go on to the pilot." if ok else
               "SMOKE TEST FAILED - do NOT run the pilot. Read cell 3 again."))

## 7. The pilot — the run that decides everything
**Problem:** is Qwen3.5-2B good enough to carry the thesis? **Why:** we must know before we
spend days on it. **In:** 30 HumanEval problems, **4 tries each**, both ways of answering.
**Out:** 240 answers saved to Drive. **Why this way:** 4 tries is the smallest number that can
show whether *some* correct answers are shorter than others — which is the whole point.

Every way of answering gets the **same** token limit (4,096) and the same settings. Only the
switch changes. That is the fairness rule in `CLAUDE.md` §4.

**If Colab disconnects, just run this cell again.** It skips what is already finished.

In [ ]:
PILOT = f"{OUT_DIR}/2026-09-21-qwen-pilot.jsonl"
for policy in ("thinking_off", "thinking_on"):
    !python scripts/gen_colab.py --policy {policy} --n 30 --samples 4 \
        --batch 16 --max-tokens 4096 --out "{PILOT}"

## 8. Grade with the benchmark's own tests
**Problem:** we must never judge an answer by eye. **Why:** that is how people fool themselves.
**In:** the answers. **Out:** a `-graded.csv`, one row per answer.
**Why this way:** each answer runs in its **own short-lived process** with a time limit, never
inside this notebook (`CLAUDE.md` §4). The tests come from the benchmark itself.

The cell first grades HumanEval's **official solutions**. They must score about 100%. If they do
not, the grader is broken and every accuracy number below would be meaningless.

In [ ]:
# Sanity check on the grader itself, using the benchmark's own correct answers.
import json, tempfile
from evalplus.data import get_human_eval_plus
import grade_humaneval as G

problems = get_human_eval_plus()
with tempfile.TemporaryDirectory() as wd:
    checked = list(problems.items())[:10]
    passed = sum(G.run_one(p["prompt"] + p["canonical_solution"], p, wd)[0] for _, p in checked)
print(f"grader sanity check: {passed}/{len(checked)} official solutions pass "
      f"{'OK' if passed == len(checked) else '<-- GRADER IS BROKEN, STOP'}")

In [ ]:
!python scripts/grade_humaneval.py --answers "{PILOT}"

## 9. The three gates
**Problem:** we must decide with numbers we fixed in advance, not with a feeling.
**Why:** otherwise we will talk ourselves into whatever the result happens to be.
**In:** the graded file. **Out:** three numbers, each PASS or FAIL.

| Gate | Must be | In plain words |
|---|---|---|
| Solved at least once | **≥ 40%** | is the model good enough to have anything to learn from? |
| Room to shorten | **≤ 0.75** | **your "will fine-tuning cut tokens?" question** — is there at least 25% to cut? |
| Cost | tokens per answer | can we finish inside free Colab? |

In [ ]:
GRADED = PILOT.replace(".jsonl", "-graded.csv")
!python scripts/check_gates.py --graded "{GRADED}" --policy thinking_on

## 10. You are here

What the result means — **decided before we looked**, so we cannot fool ourselves:

```
all gates pass          -> keep Qwen3.5-2B. Write the DECISIONS rows. Go to the full run.
solved < 40%            -> the model is too weak. Change to Qwen3.5-4B, run this notebook again.
room to shorten > 0.75  -> do NOT change model yet. Run 8 tries instead of 4 (DECISIONS #27).
                           If it still fails, that is a REAL FINDING. Write it down honestly.
too slow for free Colab -> only then buy the $10-20 package that DECISIONS #48 allows.
```

**Write the numbers down** in `results/2026-09-21-qwen-pilot.md` and add the `DECISIONS.md`
rows. A number that is not written down did not happen.

**Next:** the full run — 234 problems × 5 ways of answering × 4 tries.